# 🗂️ EnerGIS Workflow Manager

Verwalte gespeicherte Workflow-Simulationen: Umbenennen, Löschen, Notizen hinzufügen.

## 📋 Features

- **📊 Übersicht**: Alle gespeicherten Workflows in Tabellenform
- **✏️ Umbenennen**: Workflows aussagekräftig benennen
- **📝 Notizen**: Beschreibungen hinzufügen/bearbeiten
- **🗑️ Löschen**: Nicht benötigte Workflows entfernen
- **🔍 Details**: Metadaten und Konfiguration anzeigen
- **🚀 Quick Actions**: Direkt im Dashboard oder Vergleich öffnen

---

## 📦 Setup & Imports

In [ ]:
# Bootstrap: Projekt-Root finden und zum Pfad hinzufügen
import sys
from pathlib import Path

# Finde Projekt-Root (suche nach .git und energis/ Verzeichnis)
current = Path.cwd()
project_root = None

for candidate in [current] + list(current.parents):
    if (candidate / '.git').exists() and (candidate / 'energis').exists():
        project_root = candidate
        break

if project_root is None:
    # Fallback: Suche nach Projektname
    for candidate in [current] + list(current.parents):
        if candidate.name == 'Planing-Framework-for-Heat':
            project_root = candidate
            break

if project_root is None:
    project_root = current

# Füge zum Python-Pfad hinzu (damit energis importiert werden kann)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Jetzt können wir notebook_helpers importieren
try:
    from energis.io.notebook_helpers import setup_notebook_environment
    
    PROJECT_ROOT = setup_notebook_environment()
    print("\n✅ Setup abgeschlossen")
    
except ImportError as e:
    print(f"❌ Import-Fehler: {e}")
    print("\n💡 Fehlende Dependencies installieren:")
    print("   cd " + str(project_root))
    print("   pip install pandas numpy openpyxl matplotlib pyomo")
    print("\n   Oder mit allen Notebook-Dependencies:")
    print("   pip install -e .[notebooks]")
    raise

In [ ]:
# Imports
import os
import json
import shutil
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Optional

import pandas as pd
from IPython.display import display, HTML, clear_output

from energis.io.notebook_helpers import (
    list_saved_workflows,
    load_workflow_from_saved,
    create_and_display_dashboard
)

print("✅ Imports erfolgreich")

## 🔧 Helper-Funktionen

In [ ]:
def get_workflow_metadata_path(workflow_dir: Path) -> Path:
    """Get path to metadata.json for a workflow."""
    return workflow_dir / "metadata.json"


def update_workflow_metadata(workflow_dir: Path, updates: Dict) -> None:
    """Update metadata.json with new values."""
    metadata_path = get_workflow_metadata_path(workflow_dir)
    
    if not metadata_path.exists():
        raise FileNotFoundError(f"Metadata nicht gefunden: {metadata_path}")
    
    with open(metadata_path, 'r', encoding='utf-8') as f:
        metadata = json.load(f)
    
    metadata.update(updates)
    metadata['last_modified'] = datetime.now().isoformat()
    
    with open(metadata_path, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False)
    
    print(f"✅ Metadata aktualisiert: {workflow_dir.name}")


def rename_workflow(workflow_dir: Path, new_name: str) -> Path:
    """Rename a workflow directory and update metadata."""
    old_name = workflow_dir.name
    
    # Update metadata first
    update_workflow_metadata(workflow_dir, {'name': new_name})
    
    # Create new directory name with timestamp
    timestamp = old_name.split('_')[-1]  # Keep original timestamp
    new_dir_name = f"{new_name.replace(' ', '_')}_{timestamp}"
    new_path = workflow_dir.parent / new_dir_name
    
    # Rename directory
    workflow_dir.rename(new_path)
    
    print(f"✅ Workflow umbenannt: '{old_name}' → '{new_dir_name}'")
    return new_path


def delete_workflow(workflow_dir: Path, confirm: bool = True) -> None:
    """Delete a workflow directory.
    
    Args:
        workflow_dir: Path to workflow directory
        confirm: If True, require explicit confirmation
    """
    if confirm:
        print(f"⚠️  ACHTUNG: Workflow wird unwiderruflich gelöscht!")
        print(f"   Pfad: {workflow_dir}")
        response = input("   Bestätigen mit 'DELETE': ")
        
        if response != "DELETE":
            print("❌ Löschvorgang abgebrochen")
            return
    
    shutil.rmtree(workflow_dir)
    print(f"✅ Workflow gelöscht: {workflow_dir.name}")


def add_workflow_note(workflow_dir: Path, note: str, append: bool = False) -> None:
    """Add or update note in workflow metadata.
    
    Args:
        workflow_dir: Path to workflow directory
        note: Note text to add
        append: If True, append to existing note; if False, replace
    """
    metadata_path = get_workflow_metadata_path(workflow_dir)
    
    with open(metadata_path, 'r', encoding='utf-8') as f:
        metadata = json.load(f)
    
    if append and 'notes' in metadata:
        metadata['notes'] = metadata['notes'] + "\n" + note
    else:
        metadata['notes'] = note
    
    metadata['last_modified'] = datetime.now().isoformat()
    
    with open(metadata_path, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False)
    
    print(f"✅ Notiz {'hinzugefügt' if append else 'aktualisiert'}: {workflow_dir.name}")


def show_workflow_details(workflow_dir: Path) -> None:
    """Display detailed metadata for a workflow."""
    metadata_path = get_workflow_metadata_path(workflow_dir)
    
    if not metadata_path.exists():
        print(f"❌ Keine Metadata gefunden für: {workflow_dir.name}")
        return
    
    with open(metadata_path, 'r', encoding='utf-8') as f:
        metadata = json.load(f)
    
    print("="*70)
    print(f"📊 {metadata.get('name', 'Unnamed Workflow')}")
    print("="*70)
    print(f"\n📁 Verzeichnis:     {workflow_dir.name}")
    print(f"📅 Erstellt:        {metadata.get('created', 'N/A')}")
    print(f"🔄 Letzte Änderung: {metadata.get('last_modified', 'N/A')}")
    print(f"📝 Beschreibung:    {metadata.get('description', 'Keine Beschreibung')}")
    
    if 'notes' in metadata and metadata['notes']:
        print(f"\n💬 Notizen:\n{metadata['notes']}")
    
    print(f"\n🔧 Konfiguration:")
    for i, cfg in enumerate(metadata.get('config_paths', []), 1):
        print(f"   {i}. {cfg}")
    
    print(f"\n📊 Workflow-Plan:")
    for step in metadata.get('workflow_plan', {}).get('steps', []):
        print(f"   • {step}")
    
    # File sizes
    total_size = sum(f.stat().st_size for f in workflow_dir.rglob('*') if f.is_file())
    print(f"\n💾 Speichergröße:   {total_size / 1024 / 1024:.2f} MB")
    
    # Count files
    n_pkl = len(list(workflow_dir.glob('*.pkl')))
    n_csv = len(list(workflow_dir.rglob('*.csv')))
    n_pdf = len(list(workflow_dir.rglob('*.pdf')))
    n_svg = len(list(workflow_dir.rglob('*.svg')))
    
    print(f"\n📄 Dateien:")
    print(f"   PKL: {n_pkl}, CSV: {n_csv}, PDF: {n_pdf}, SVG: {n_svg}")
    print("="*70)

print("✅ Helper-Funktionen geladen")

## 📊 Workflow-Übersicht

Zeigt alle gespeicherten Workflows in Tabellenform.

In [ ]:
# Workflows laden
workflows = list_saved_workflows(sort_by="date")

if not workflows:
    print("❌ Keine gespeicherten Workflows gefunden!")
    print("\n💡 Tipp: Führe erst einen Optimierungslauf in runner.ipynb oder scenario_studio.ipynb aus.")
else:
    # Als DataFrame anzeigen
    df_data = []
    for i, wf in enumerate(workflows, 1):
        df_data.append({
            '#': i,
            'Name': wf['name'],
            'Erstellt': wf['created_str'],
            'Workflow': ' → '.join(wf['steps']),
            'Beschreibung': wf['description'][:50] + '...' if len(wf['description']) > 50 else wf['description'],
            'Pfad': wf['path'].name,
        })
    
    df = pd.DataFrame(df_data)
    
    print(f"\n📋 {len(workflows)} gespeicherte Workflow(s):\n")
    display(df)
    
    print("\n💡 Tipp: Nutze die Zellen unten für Workflow-Management")

## ✏️ Workflow umbenennen

Benenne einen Workflow um.

In [ ]:
# Konfiguration
WORKFLOW_INDEX = 1  # Index aus Tabelle oben (Spalte '#')
NEW_NAME = "Optimized Baseline 2024"

# Umbenennen
if workflows and 1 <= WORKFLOW_INDEX <= len(workflows):
    workflow_meta = workflows[WORKFLOW_INDEX - 1]
    old_name = workflow_meta['name']
    
    print(f"🔄 Benenne um: '{old_name}' → '{NEW_NAME}'\n")
    
    new_path = rename_workflow(workflow_meta['path'], NEW_NAME)
    
    print(f"\n✅ Erfolgreich umbenannt!")
    print(f"   Neuer Pfad: {new_path}")
    
    print("\n💡 Tipp: Führe die Übersichts-Zelle erneut aus, um die Änderung zu sehen.")
else:
    print("❌ Ungültiger WORKFLOW_INDEX. Bitte Wert zwischen 1 und", len(workflows), "wählen.")

## 📝 Beschreibung/Notizen bearbeiten

Füge Notizen zu einem Workflow hinzu oder bearbeite die Beschreibung.

In [ ]:
# Konfiguration
WORKFLOW_INDEX = 1  # Index aus Tabelle oben
NOTE = """Baseline-Simulation mit optimierten Parametern.
- CO2-Preis: 80 EUR/t
- Wärmepumpen-Kapazität: 5 MW
- Speicher aktiviert
"""
APPEND = False  # True = an bestehende Notiz anhängen, False = ersetzen

# Notiz hinzufügen
if workflows and 1 <= WORKFLOW_INDEX <= len(workflows):
    workflow_meta = workflows[WORKFLOW_INDEX - 1]
    
    print(f"📝 Füge Notiz hinzu zu: {workflow_meta['name']}\n")
    
    add_workflow_note(workflow_meta['path'], NOTE, append=APPEND)
    
    print("\n✅ Notiz gespeichert!")
    print("\n💡 Tipp: Nutze 'Details anzeigen' unten, um die Notiz zu sehen.")
else:
    print("❌ Ungültiger WORKFLOW_INDEX. Bitte Wert zwischen 1 und", len(workflows), "wählen.")

## 🔍 Workflow-Details anzeigen

Zeige vollständige Metadaten und Statistiken für einen Workflow.

In [ ]:
# Konfiguration
WORKFLOW_INDEX = 1  # Index aus Tabelle oben

# Details anzeigen
if workflows and 1 <= WORKFLOW_INDEX <= len(workflows):
    workflow_meta = workflows[WORKFLOW_INDEX - 1]
    show_workflow_details(workflow_meta['path'])
else:
    print("❌ Ungültiger WORKFLOW_INDEX. Bitte Wert zwischen 1 und", len(workflows), "wählen.")

## 🗑️ Workflow löschen

**⚠️ VORSICHT**: Dieser Vorgang ist unwiderruflich!

In [ ]:
# Konfiguration
WORKFLOW_INDEX = 1  # Index aus Tabelle oben
REQUIRE_CONFIRMATION = True  # Auf False setzen um Bestätigung zu überspringen (NICHT EMPFOHLEN!)

# Löschen
if workflows and 1 <= WORKFLOW_INDEX <= len(workflows):
    workflow_meta = workflows[WORKFLOW_INDEX - 1]
    
    print(f"🗑️  Lösche Workflow: {workflow_meta['name']}\n")
    
    delete_workflow(workflow_meta['path'], confirm=REQUIRE_CONFIRMATION)
    
    print("\n💡 Tipp: Führe die Übersichts-Zelle erneut aus, um die Änderung zu sehen.")
else:
    print("❌ Ungültiger WORKFLOW_INDEX. Bitte Wert zwischen 1 und", len(workflows), "wählen.")

## 🚀 Quick Actions

Schnellzugriff auf häufige Aktionen.

### 📊 Workflow im Dashboard öffnen

In [ ]:
# Konfiguration
WORKFLOW_INDEX = 1  # Index aus Tabelle oben

# Dashboard öffnen
if workflows and 1 <= WORKFLOW_INDEX <= len(workflows):
    workflow_meta = workflows[WORKFLOW_INDEX - 1]
    
    print(f"📊 Lade Dashboard für: {workflow_meta['name']}\n")
    
    # Workflow laden
    workflow = load_workflow_from_saved(workflow_meta['path'], verbose=True)
    
    # Dashboard erstellen
    try:
        dashboard = create_and_display_dashboard(
            workflow,
            title=f"Workflow Manager - {workflow_meta['name']}"
        )
        
        print("\n✅ Dashboard geladen!\n")
        
        # Dashboard anzeigen
        dashboard
        
    except ImportError as e:
        print(f"❌ Dashboard-Import-Fehler: {e}")
        print("   Installation: pip install panel holoviews bokeh plotly")
else:
    print("❌ Ungültiger WORKFLOW_INDEX. Bitte Wert zwischen 1 und", len(workflows), "wählen.")

### 🔄 Batch-Operationen

Führe Operationen auf mehreren Workflows gleichzeitig aus.

In [ ]:
# Batch-Operation: Notiz zu mehreren Workflows hinzufügen
WORKFLOW_INDICES = [1, 2, 3]  # Indices aus Tabelle oben
BATCH_NOTE = "Baseline-Serie für Publikation"
APPEND = True

print(f"🔄 Batch-Operation: Notiz zu {len(WORKFLOW_INDICES)} Workflows hinzufügen\n")

for idx in WORKFLOW_INDICES:
    if workflows and 1 <= idx <= len(workflows):
        workflow_meta = workflows[idx - 1]
        print(f"   {idx}. {workflow_meta['name']}...")
        try:
            add_workflow_note(workflow_meta['path'], BATCH_NOTE, append=APPEND)
        except Exception as e:
            print(f"      ❌ Fehler: {e}")
    else:
        print(f"   ❌ Index {idx} ungültig (max: {len(workflows)})")

print("\n✅ Batch-Operation abgeschlossen!")

### 🧹 Cleanup: Alte Workflows löschen

Lösche Workflows, die älter als X Tage sind.

In [ ]:
# Konfiguration
MAX_AGE_DAYS = 30  # Workflows älter als X Tage löschen
DRY_RUN = True  # True = nur anzeigen, nicht löschen

from datetime import datetime, timedelta

cutoff_date = datetime.now() - timedelta(days=MAX_AGE_DAYS)

print(f"🧹 Suche Workflows älter als {MAX_AGE_DAYS} Tage (vor {cutoff_date.strftime('%Y-%m-%d')})\n")

old_workflows = []
for wf in workflows:
    created = datetime.fromisoformat(wf['created'])
    if created < cutoff_date:
        old_workflows.append(wf)

if not old_workflows:
    print("✅ Keine alten Workflows gefunden.")
else:
    print(f"📋 {len(old_workflows)} alte(r) Workflow(s) gefunden:\n")
    
    for wf in old_workflows:
        print(f"   • {wf['name']} ({wf['created_str']})")
    
    if DRY_RUN:
        print("\n💡 DRY_RUN = True: Workflows werden NICHT gelöscht")
        print("   Setze DRY_RUN = False und führe die Zelle erneut aus, um zu löschen.")
    else:
        print("\n⚠️  ACHTUNG: Workflows werden jetzt gelöscht!")
        response = input("   Bestätigen mit 'DELETE ALL': ")
        
        if response == "DELETE ALL":
            for wf in old_workflows:
                try:
                    delete_workflow(wf['path'], confirm=False)
                except Exception as e:
                    print(f"   ❌ Fehler beim Löschen von {wf['name']}: {e}")
            print("\n✅ Cleanup abgeschlossen!")
        else:
            print("\n❌ Cleanup abgebrochen")

---

## 📚 Weitere Informationen

### Verwandte Notebooks:
- **runner.ipynb** - Optimierungsläufe durchführen
- **scenario_studio.ipynb** - Interaktive Szenario-Analyse
- **interactive_dashboard.ipynb** - Dashboard für gespeicherte Workflows
- **comparison_dashboard.ipynb** - Mehrere Workflows vergleichen

### Workflow-Verzeichnisstruktur:
```
saved_workflows/
├── Workflow_Name_20240101_120000/
│   ├── workflow.pkl              # Vollständiges Workflow-Objekt
│   ├── metadata.json             # Metadaten (Name, Beschreibung, etc.)
│   ├── pf_result.csv             # Perfect Forecast Ergebnisse
│   ├── rh_result.csv             # Rolling Horizon Ergebnisse
│   ├── plots/
│   │   ├── heat_balance.pdf
│   │   ├── heat_balance.svg
│   │   └── ...
│   └── configs/                  # Verwendete Konfigurationsdateien
```

### Tipps:
- Führe die Übersichts-Zelle nach Änderungen erneut aus
- Nutze aussagekräftige Namen für bessere Übersicht
- Füge Notizen hinzu für Dokumentation
- Regelmäßiges Cleanup spart Speicherplatz

---